# 章节实践解答（06.07）

本解答对应章节 06.07_chapter_test.ipynb 中的综合编程实践题：补全 fuse_conv_bn_eval 与 fuse_model，验证正确性并统计算子数量。

In [ ]:
import copy
import torch
import torch.nn as nn


def fuse_conv_bn_eval(conv, bn):
    """将推理阶段的 BatchNorm 折叠进 Conv2d。"""
    fused = nn.Conv2d(
        conv.in_channels, conv.out_channels, conv.kernel_size,
        stride=conv.stride, padding=conv.padding, groups=conv.groups,
        bias=True, device=conv.weight.device, dtype=conv.weight.dtype)
    with torch.no_grad():
        fused.weight.data.copy_(conv.weight)
        bias = conv.bias if conv.bias is not None else torch.zeros_like(bn.bias)
        scale = bn.weight / torch.sqrt(bn.running_var + bn.eps)
        fused.weight.data.mul_(scale.reshape(-1, 1, 1, 1))
        fused.bias.data.copy_((bias - bn.running_mean) * scale + bn.bias)
    return fused


def fuse_model(model):
    """递归折叠模型中所有 Conv+BN 结构（仅推理）。"""
    model = copy.deepcopy(model).eval()
    for name, child in model.named_children():
        if (isinstance(child, nn.Sequential) and len(child) >= 2
                and isinstance(child[0], nn.Conv2d)
                and isinstance(child[1], nn.BatchNorm2d)):
            fused_conv = fuse_conv_bn_eval(child[0], child[1])
            remaining = [m for m in list(child)[2:]]
            setattr(model, name, nn.Sequential(fused_conv, *remaining))
        elif isinstance(child, nn.Module):
            setattr(model, name, fuse_model(child))
    return model

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
from src.mobilenet_v3 import MobileNetV3
from src.operator_utils import count_modules

device = 'npu:0' if getattr(torch, 'npu', None) and torch.npu.is_available() else 'cpu'
model = MobileNetV3(model_mode="LARGE", num_classes=200).eval()
fused_model = fuse_model(model).to(device).eval()
model = model.to(device).eval()

x = torch.randn(8, 3, 224, 224, device=device)
with torch.no_grad():
    y_before = model(x)
    y_after = fused_model(x)

max_diff = (y_before - y_after).abs().max().item()
print("max_diff =", max_diff)
assert max_diff < 1e-2, "融合结果不一致"
print("融合前:", count_modules(model))
print("融合后:", count_modules(fused_model))

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px;">点击：查看/折叠代码说明</summary>
  <div style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li>在 fuse_model 中，递归结果必须通过 setattr(model, name, fuse_model(child)) 写回父模块，否则深层 MobileBlock 中的 BN 不会被真正移除。</li>
      <li>写权重使用 with torch.no_grad(): fused.weight.data.copy_(conv.weight)，避免对叶子参数做 in-place 操作报错。</li>
    </ul>
  </div>
</details>